In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Auto-detect where the CSVs are
current = Path(os.getcwd())
print("Current directory:", current)

# Search for the CSV in current folder and all parent folders
csv_name = "01_fund_master.csv"
found = list(current.rglob(csv_name))

if found:
    RAW = found[0].parent
    print("Found CSVs in:", RAW)
else:
    print("Not found. Searching Downloads and Desktop...")
    for candidate in [
        Path.home() / "Downloads",
        Path.home() / "Desktop",
        Path.home() / "Documents",
    ]:
        if (candidate / csv_name).exists():
            RAW = candidate
            print("Found in:", RAW)
            break
    else:
        print("Still not found — manually set RAW below")

print("\nCSVs found:")
for f in sorted(RAW.glob("*.csv")):
    print(" ", f.name)

Current directory: C:\Users\Hp
Found CSVs in: C:\Users\Hp

CSVs found:
  01_fund_master.csv
  02_nav_history.csv
  03_aum_by_fund_house.csv
  04_monthly_sip_inflows.csv
  05_category_inflows.csv
  06_industry_folio_count.csv
  07_scheme_performance.csv
  08_investor_transactions.csv
  09_portfolio_holdings.csv
  10_benchmark_indices.csv


In [2]:
df_fund     = pd.read_csv(RAW / "01_fund_master.csv")
df_nav      = pd.read_csv(RAW / "02_nav_history.csv")
df_aum      = pd.read_csv(RAW / "03_aum_by_fund_house.csv")
df_sip      = pd.read_csv(RAW / "04_monthly_sip_inflows.csv")
df_cat      = pd.read_csv(RAW / "05_category_inflows.csv")
df_folio    = pd.read_csv(RAW / "06_industry_folio_count.csv")
df_perf     = pd.read_csv(RAW / "07_scheme_performance.csv")
df_txn      = pd.read_csv(RAW / "08_investor_transactions.csv")
df_holdings = pd.read_csv(RAW / "09_portfolio_holdings.csv")
df_bench    = pd.read_csv(RAW / "10_benchmark_indices.csv")

datasets = {
    "fund_master": df_fund, "nav_history": df_nav, "aum": df_aum,
    "sip": df_sip, "category_inflows": df_cat, "folio": df_folio,
    "performance": df_perf, "transactions": df_txn,
    "holdings": df_holdings, "benchmark": df_bench
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols")

fund_master: 40 rows x 15 cols
nav_history: 46000 rows x 3 cols
aum: 90 rows x 5 cols
sip: 48 rows x 6 cols
category_inflows: 144 rows x 3 cols
folio: 21 rows x 6 cols
performance: 40 rows x 19 cols
transactions: 32778 rows x 13 cols
holdings: 322 rows x 8 cols
benchmark: 8050 rows x 3 cols


In [3]:
nav_codes  = set(df_nav["amfi_code"].unique())
fund_codes = set(df_fund["amfi_code"].unique())

missing_in_nav  = fund_codes - nav_codes
missing_in_fund = nav_codes - fund_codes

print("Codes in fund_master but missing in nav_history:", missing_in_nav)
print("Codes in nav_history but missing in fund_master:", missing_in_fund)
print(f"\nTotal schemes in master: {len(fund_codes)}")
print(f"Total schemes in nav history: {len(nav_codes)}")

Codes in fund_master but missing in nav_history: set()
Codes in nav_history but missing in fund_master: set()

Total schemes in master: 40
Total schemes in nav history: 40


In [4]:
print("=== Fund Houses ===")
print(df_fund["fund_house"].value_counts())

print("\n=== Categories ===")
print(df_fund["category"].value_counts())

print("\n=== Risk Grades ===")
print(df_fund["risk_category"].value_counts())

print("\n=== Expense Ratio Range ===")
print("Min:", df_fund["expense_ratio_pct"].min())
print("Max:", df_fund["expense_ratio_pct"].max())
print("Avg:", round(df_fund["expense_ratio_pct"].mean(), 2))


=== Fund Houses ===
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3
Name: count, dtype: int64

=== Categories ===
category
Equity    34
Debt       6
Name: count, dtype: int64

=== Risk Grades ===
risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

=== Expense Ratio Range ===
Min: 0.55
Max: 1.64
Avg: 1.24


In [5]:
import requests

SCHEMES = {
    "HDFC Top 100":   125497,
    "SBI Bluechip":   119551,
    "ICICI Bluechip": 120503,
    "Axis Bluechip":  119092,
    "Kotak Bluechip": 120841,
}

for name, code in SCHEMES.items():
    try:
        url = f"https://api.mfapi.in/mf/{code}"
        r = requests.get(url, timeout=10)
        data = r.json()
        df_live = pd.DataFrame(data["data"])
        df_live["amfi_code"] = code
        df_live.to_csv(RAW / f"live_nav_{code}.csv", index=False)
        print(f"{name}: {len(df_live)} rows | Latest NAV = {df_live['nav'].iloc[0]}")
    except Exception as e:
        print(f"{name}: FAILED — {e}")
        

HDFC Top 100: 3095 rows | Latest NAV = 194.75470
SBI Bluechip: 3240 rows | Latest NAV = 105.10700
ICICI Bluechip: 3311 rows | Latest NAV = 103.81020
Axis Bluechip: 3569 rows | Latest NAV = 6168.47550
Kotak Bluechip: 3305 rows | Latest NAV = 246.08780
